# 02 — Advanced Classification
**ITAI 2373 | Leroy Brown | Houston Community College**

Re-train the 97.6%-accuracy Logistic Regression classifier, run cross-validation, compare classifiers, and visualize a confusion matrix.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub, glob

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

from src.data_processing.text_preprocessor import preprocess
from src.data_processing.feature_extractor import fit_tfidf
from src.data_processing.data_validator import clean_dataframe
from src.analysis.classifier import NewsClassifier
from src.analysis.sentiment_analyzer import score_corpus, sentiment_by_category
from src.utils.evaluation import full_classification_report, cross_validate_classifier, compare_classifiers, confusion_matrix_df
from config.settings import CATEGORIES, DATASET_SIZE, RANDOM_STATE

print("Imports complete.")

## 1. Load & Prepare Data

In [ ]:
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df_raw.columns = df_raw.columns.str.lower().str.strip()
if "category" not in df_raw.columns:
    for col in df_raw.columns:
        if df_raw[col].nunique() <= 10:
            df_raw.rename(columns={col: "category"}, inplace=True)
            break
text_col = [c for c in df_raw.columns if any(k in c for k in ["text","content","article"])][0]
df_raw.rename(columns={text_col: "text"}, inplace=True)

df = clean_dataframe(df_raw)
df = df[df["category"].isin(CATEGORIES)]
df = df.sample(n=min(DATASET_SIZE, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
df["clean"] = df["text"].apply(preprocess)
print(f"Dataset ready: {len(df)} articles")

## 2. Fit TF-IDF & Train/Test Split

In [ ]:
tfidf, X = fit_tfidf(df["clean"])
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 3. Train Logistic Regression (Best Model)

In [ ]:
clf = NewsClassifier(vectorizer=tfidf)
results = clf.train_test_evaluate(X, y, test_size=0.2)

print(f"Accuracy: {results['accuracy']:.4f}")
print()
print(results["report"])

## 4. Cross-Validation (5-Fold Stratified)

In [ ]:
cv_results = cross_validate_classifier(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), X, y, cv=5)
print(f"CV Mean Accuracy: {cv_results['mean_accuracy']:.4f} ± {cv_results['std']:.4f}")
print(f"Fold scores: {[round(s,4) for s in cv_results['scores']]}")

## 5. Classifier Comparison

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "MultinomialNB":       MultinomialNB(),
    "LinearSVC":          LinearSVC(max_iter=2000, random_state=RANDOM_STATE),
}
comparison = compare_classifiers(classifiers, X_train, X_test, y_train, y_test)
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(comparison["classifier"], comparison["accuracy"], color=["#1f77b4","#ff7f0e","#2ca02c"])
ax.set_ylim(0.9, 1.0)
ax.set_title("Classifier Accuracy Comparison", fontweight="bold")
ax.set_ylabel("Accuracy")
for i, row in comparison.iterrows():
    ax.text(i, row["accuracy"] + 0.001, f'{row["accuracy"]:.4f}', ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("../data/results/classifier_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Confusion Matrix

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)
preds = lr.predict(X_test)

cm = confusion_matrix_df(y_test, preds, labels=CATEGORIES)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix — Logistic Regression", fontweight="bold")
ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("../data/results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Enhanced Sentiment Analysis by Category

In [ ]:
sent_df = score_corpus(df["text"])
df = pd.concat([df, sent_df], axis=1)
print(sentiment_by_category(df).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
sentiment_by_category(df)[["sent_pos","sent_neg","sent_neu"]].plot(
    kind="bar", ax=ax, color=["#2ca02c","#d62728","#aec7e8"]
)
ax.set_title("Sentiment Dimensions by Category", fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Mean Score")
ax.tick_params(axis="x", rotation=30)
ax.legend(["Positive","Negative","Neutral"])
plt.tight_layout()
plt.savefig("../data/results/sentiment_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Classifier | Accuracy |
|------------|----------|
| Logistic Regression | **97.6%** |
| LinearSVC | ~97.6% |
| MultinomialNB | ~96.x% |

- Politics has the lowest F1 (0.957) due to overlap with Business topics
- Sport achieves the highest F1 (0.991) — most semantically distinct vocabulary

**Next:** `03_Topic_Modeling.ipynb`